# Chapter 4 Lab: Tight Frames

This notebook accompanies **Chapter 4**. It builds the regular-polygon
and projection-construction machinery for tight frames, then works
through all five lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def frame_operator(F):
    return F @ F.T

def exact_frame_bounds(F):
    S = frame_operator(F)
    eigenvalues, eigenvectors = np.linalg.eigh(S)
    return float(eigenvalues[0]), float(eigenvalues[-1]), eigenvalues, eigenvectors

def is_tight(F, tol=1e-9):
    A, B, _, _ = exact_frame_bounds(F)
    return (B - A) < tol * max(1.0, B)

def regular_polygon_frame(m):
    angles = 2 * np.pi * np.arange(m) / m
    return np.vstack([np.cos(angles), np.sin(angles)])

## Lab Exercise 4.1: The Full Regular Polygon Table

Build a table of $A$, $B$, and $\operatorname{tr}(S)$ for $m=3,4,\ldots,20$. Verify (i) $A=B$, (ii) $A=m/2$, (iii) $\operatorname{tr}(S)=m$. Then check $m=2$ separately.

In [ ]:
print(f"{'m':>3} {'A':>8} {'B':>8} {'tr(S)':>8} {'tight':>7} {'A=m/2':>7} {'tr=m':>6}")
for m in range(3, 21):
    F = regular_polygon_frame(m)
    A, B, _, _ = None, None, None, None  # TODO: exact_frame_bounds(F)
    trace_S = None  # TODO: np.trace(frame_operator(F))

    tight_ok = np.isclose(A, B, atol=1e-9)
    Am2_ok = np.isclose(A, m/2, atol=1e-9)
    trm_ok = np.isclose(trace_S, m, atol=1e-9)
    print(f"{m:>3} {A:>8.4f} {B:>8.4f} {trace_S:>8.4f} {str(tight_ok):>7} {str(Am2_ok):>7} {str(trm_ok):>6}")

print("\n--- m=2 (expected to break the pattern) ---")
F2 = regular_polygon_frame(2)
A2, B2, eigs2, _ = exact_frame_bounds(F2)
print(f"m=2: A={A2:.4f}, B={B2:.4f}, tight={np.isclose(A2,B2)}, eigenvalues={eigs2}")

## Lab Exercise 4.2: Verifying the Projection Construction

Implement `projection\_parseval\_frame` and reproduce Example 4.2. Then construct a second example: $V\subset\mathbb{R}^4$, 2-dimensional (via `np.linalg.qr`), and verify projecting $\{e_1,\ldots,e_4\}$ onto $V$ gives a Parseval frame.

In [ ]:
def projection_parseval_frame(N, V_basis):
    '''Given orthonormal basis columns of V (shape (N,n)), return the Parseval frame.'''
    return V_basis.T

v1 = np.array([1., -1., 0.]) / np.sqrt(2)
v2 = np.array([1., 1., -2.]) / np.sqrt(6)
V_basis_3d = np.column_stack([v1, v2])

F_proj_3d = None  # TODO: projection_parseval_frame(3, V_basis_3d)
A_3d, B_3d, _, _ = None, None, None, None  # TODO: exact_frame_bounds(F_proj_3d)

print(f"Example 4.2 reproduction: A={A_3d:.6f}, B={B_3d:.6f}  (expect A=B=1)")

rng = np.random.default_rng(7)
random_vecs = rng.standard_normal((4, 2))
Q, _ = np.linalg.qr(random_vecs)
V_basis_4d = Q

F_proj_4d = None  # TODO: projection_parseval_frame(4, V_basis_4d)
A_4d, B_4d, _, _ = None, None, None, None  # TODO: exact_frame_bounds(F_proj_4d)

print(f"R^4 example: A={A_4d:.6f}, B={B_4d:.6f}  (expect A=B=1)")

## Lab Exercise 4.3: Partial Regular Polygons Are Not Tight

For $m=8$, compute bounds for the first $m'=2,\ldots,7$ vertices only. For which $m'$ is the frame still tight? Reconcile with the roots-of-unity proof.

In [ ]:
m_full = 8
all_vertices = regular_polygon_frame(m_full)

print(f"{'mprime':>7} {'A':>8} {'B':>8} {'tight':>7}")
for m_prime in range(2, 8):
    F_partial = all_vertices[:, :m_prime]
    A_p, B_p, _, _ = None, None, None, None  # TODO: exact_frame_bounds(F_partial)
    print(f"{m_prime:>7} {A_p:>8.4f} {B_p:>8.4f} {str(np.isclose(A_p,B_p)):>7}")

*Your reconciliation: which step of the roots-of-unity proof breaks down for a partial set?* (replace this text)

## Lab Exercise 4.4: Hunting for Asymmetric Tight Frames

Starting from $g_1=g_3=(1,0)$, $g_2=g_4=(0,1)$, perturb by small rotation/rescaling; check if still tight. Then construct a genuinely asymmetric exact tight frame.

In [ ]:
g1 = np.array([1.0, 0.0]); g2 = np.array([0.0, 1.0])
g3 = np.array([1.0, 0.0]); g4 = np.array([0.0, 1.0])
F_repeated = np.column_stack([g1, g2, g3, g4])
A0, B0, _, _ = exact_frame_bounds(F_repeated)
print(f"Before perturbation: A={A0:.4f}, B={B0:.4f}, B/A={B0/A0:.6f}")

rng = np.random.default_rng(3)
perturbed_vectors = []
for g in [g1, g2, g3, g4]:
    angle_pert = None  # TODO: rng.uniform(-0.05, 0.05)
    scale_pert = None  # TODO: 1 + rng.uniform(-0.1, 0.1)
    rot = np.array([[np.cos(angle_pert), -np.sin(angle_pert)],
                     [np.sin(angle_pert),  np.cos(angle_pert)]])
    perturbed_vectors.append(scale_pert * (rot @ g))

F_perturbed = np.column_stack(perturbed_vectors)
A1, B1, _, _ = exact_frame_bounds(F_perturbed)
print(f"After perturbation:  A={A1:.4f}, B={B1:.4f}, B/A={B1/A1:.6f}")

def tightness_violation(params):
    vecs = []
    for i in range(4):
        r, theta = params[2*i], params[2*i+1]
        vecs.append(np.array([r*np.cos(theta), r*np.sin(theta)]))
    F = np.column_stack(vecs)
    S = frame_operator(F)
    A_target = np.trace(S) / 2
    return np.sum((S - A_target*np.eye(2))**2)

x0 = np.array([1.0, 0.3, 1.2, 1.1, 0.9, 2.4, 1.05, 4.0])
result = minimize(tightness_violation, x0, method='Nelder-Mead')
print(f"\nOptimization result (should be near 0): {result.fun:.2e}")

final_vecs = []
for i in range(4):
    r, theta = result.x[2*i], result.x[2*i+1]
    final_vecs.append(np.array([r*np.cos(theta), r*np.sin(theta)]))
F_final = np.column_stack(final_vecs)
print("Final vectors:\n", F_final)
print("Frame operator S:\n", frame_operator(F_final))

*Report the B/A ratio before/after perturbation, and describe your asymmetric tight frame (if you found one):* (replace this text)

## Lab Exercise 4.5 (Optional, Challenge): Roots of Unity in C^n

For $n=4$: (a) build $f_k=(1/\sqrt n)(1,\omega^k,\ldots,\omega^{(n-1)k})$, verify orthonormality via `np.vdot`. (b) drop the last vector; is the remaining 3-vector set a frame for $\mathbb{C}^4$?

In [ ]:
n = 4
omega = np.exp(2j * np.pi / n)

F_dft = np.zeros((n, n), dtype=complex)
for k in range(n):
    F_dft[:, k] = None  # TODO: (1/sqrt(n)) * array of omega^(j*k) for j in range(n)

print("Inner product matrix <f_j, f_k> (should be the identity):")
gram = np.zeros((n, n), dtype=complex)
for j in range(n):
    for k in range(n):
        gram[j, k] = None  # TODO: np.vdot(F_dft[:,j], F_dft[:,k])
print(gram.round(6))
print(f"Is identity? {np.allclose(gram, np.eye(n))}")

F_partial = F_dft[:, :n-1]
rank_partial = np.linalg.matrix_rank(F_partial)
print(f"\nDropping f_{{n-1}}: remaining set has rank {rank_partial} in C^{n}")
print(f"Spans C^{n}? {rank_partial == n}")

*Your answer for part (b): why can 3 vectors never be a frame for C^4, by Theorem 2.2.1?* (replace this text)